# Preprocessing & Modeling — Sistem Rekomendasi Wisata Bandung

**Nama**: Mira Aldina  
**NIM**: 41037006221007  
**Prodi**: Teknik Informatika - UNINUS  

**Metode**: Content-Based Filtering dengan TF-IDF + Cosine Similarity  
**Dataset**: 331 destinasi wisata Bandung Raya (setelah hapus duplikat)

## Tahapan:
1. Load dataset CSV
2. Text Preprocessing (Case Folding, Cleaning, Tokenization, Stopword Removal, Stemming)
3. Pembobotan TF-IDF
4. Perhitungan Cosine Similarity
5. Test rekomendasi
6. Save model (.pkl) untuk dipakai backend Flask

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import re
import pickle
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print('Semua library berhasil di-import')

Semua library berhasil di-import


## 2. Load Dataset

In [2]:
# Load dataset CSV
df = pd.read_csv('../data/dataset_wisata_clean.csv')

print(f'Total destinasi: {len(df)}')
print(f'Kolom: {df.columns.tolist()}')
df.head()

Total destinasi: 331
Kolom: ['Place_Id', 'Place_Name', 'Description', 'Category', 'City']


,Place_Id,Place_Name,Description,Category,City
0,1,Alun-Alun Bandung,Alun-alun pusat kota Bandung dengan taman rump...,Taman Kota,Kota Bandung
1,2,Gedung Sate,Bangunan bersejarah peninggalan kolonial Belan...,Wisata Sejarah,Kota Bandung
2,3,Jalan Braga,Jalan bersejarah dengan deretan bangunan kolon...,Wisata Sejarah,Kota Bandung
3,4,Trans Studio Bandung,Taman hiburan indoor terbesar di Bandung denga...,Taman Hiburan,Kota Bandung
4,5,Saung Angklung Udjo,Pusat budaya Sunda dengan pertunjukan angklung...,Wisata Budaya,Kota Bandung


In [3]:
# Cek info dataset
print('=== INFO DATASET ===')
print(df.info())
print('\n=== JUMLAH PER KATEGORI ===')
print(df['Category'].value_counts())
print('\n=== JUMLAH PER WILAYAH ===')
print(df['City'].value_counts())

=== INFO DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 331 entries, 0 to 330
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Place_Id     331 non-null    int64 
 1   Place_Name   331 non-null    object
 2   Description  331 non-null    object
 3   Category     331 non-null    object
 4   City         331 non-null    object
dtypes: int64(1), object(4)
memory usage: 13.1+ KB
None

=== JUMLAH PER KATEGORI ===
Category
Wisata Alam           157
Taman Kota             36
Pusat Perbelanjaan     21
Wisata Kuliner         19
Taman Hiburan          18
Wisata Sejarah         16
Wisata Olahraga        16
Wisata Edukasi         14
Museum                 10
Wisata Budaya           8
Wisata Religi           8
Wisata Seni             5
Desa Wisata             3
Name: count, dtype: int64

=== JUMLAH PER WILAYAH ===
City
Kabupaten Bandung    126
Kota Bandung         118
Bandung Barat         87
Name: count, dtype: in

In [4]:
# Cek apakah ada data kosong
print('=== CEK NULL ===')
print(df.isnull().sum())

# Cek duplikasi nama
duplicates = df[df.duplicated(subset=['Place_Name'], keep=False)]
print(f'\nTotal nama duplikat: {len(duplicates)}')
if len(duplicates) > 0:
    print(duplicates[['Place_Id', 'Place_Name']].head(10))

=== CEK NULL ===
Place_Id       0
Place_Name     0
Description    0
Category       0
City           0
dtype: int64

Total nama duplikat: 0


## 3. Text Preprocessing

Tahapan preprocessing:
1. **Case Folding** - ubah semua huruf ke lowercase
2. **Cleaning** - hapus tanda baca, angka, simbol
3. **Tokenization** - pecah teks jadi kata-kata
4. **Stopword Removal** - hapus kata umum (di, yang, dan, dll)
5. **Stemming** - ubah kata berimbuhan jadi kata dasar (Sastrawi)

**PENTING**: Fungsi preprocess di notebook ini harus SAMA PERSIS dengan yang di backend `app.py`!

In [5]:
# Setup Sastrawi
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopwords_id = stopword_factory.get_stop_words()

print(f'Sastrawi stemmer ready')
print(f'Total stopwords Bahasa Indonesia: {len(stopwords_id)}')
print(f'Contoh stopwords: {stopwords_id[:10]}')

Sastrawi stemmer ready
Total stopwords Bahasa Indonesia: 126
Contoh stopwords: ['yang', 'untuk', 'pada', 'ke', 'para', 'namun', 'menurut', 'antara', 'dia', 'dua']


In [6]:
def preprocess_text(text):
    # 1. Case folding
    text = text.lower()
    
    # 2. Cleaning - hapus karakter non-alfabet
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 3. Tokenization
    tokens = text.split()
    
    # 4. Stopword removal
    tokens = [word for word in tokens if word not in stopwords_id]
    
    # 5. Stemming per kata
    tokens = [stemmer.stem(word) for word in tokens]
    
    return ' '.join(tokens)

# Test preprocessing
sample = df.iloc[0]
print(f'Original: {sample["Description"]}')
print(f'Processed: {preprocess_text(sample["Description"])}')

Original: Alun-alun pusat kota Bandung dengan taman rumput sintetis luas, area bersantai keluarga, dan ikon ruang publik perkotaan.
Processed: alunalun pusat kota bandung taman rumput sintetis luas area santa keluarga ikon ruang publik kota


In [7]:
# Contoh tahapan preprocessing detail (untuk laporan BAB III/IV)
def preprocess_text_detailed(text):
    """Versi detail untuk menampilkan output tiap tahapan"""
    print(f'TEKS ASLI:')
    print(f'  -> {text}\n')
    
    step1 = text.lower() # 1. Case folding
    print(f'1. CASE FOLDING:')
    print(f'  -> {step1}\n')
    
    step2 = re.sub(r'[^a-z\s]', '', step1)  # 2. Cleaning
    print(f'2. CLEANING:')
    print(f'  -> {step2}\n')
    
    step3 = step2.split() # 3. Tokenization
    print(f'3. TOKENIZATION:')
    print(f'  -> {step3}\n')
    
    step4 = [t for t in step3 if t not in stopwords_id] # 4. Stopword removal
    print(f'4. STOPWORD REMOVAL:')
    print(f'  -> {step4}\n')
    
    step5 = [stemmer.stem(w) for w in step4] # 5. Stemming
    print(f'5. STEMMING:')
    print(f'  -> {step5}\n')
    
    return ' '.join(step5)

sample_text = 'Kawah Putih adalah wisata alam di Bandung yang sangat indah dan populer!'
result = preprocess_text_detailed(sample_text)

TEKS ASLI:
  -> Kawah Putih adalah wisata alam di Bandung yang sangat indah dan populer!

1. CASE FOLDING:
  -> kawah putih adalah wisata alam di bandung yang sangat indah dan populer!

2. CLEANING:
  -> kawah putih adalah wisata alam di bandung yang sangat indah dan populer

3. TOKENIZATION:
  -> ['kawah', 'putih', 'adalah', 'wisata', 'alam', 'di', 'bandung', 'yang', 'sangat', 'indah', 'dan', 'populer']

4. STOPWORD REMOVAL:
  -> ['kawah', 'putih', 'wisata', 'alam', 'bandung', 'sangat', 'indah', 'populer']

5. STEMMING:
  -> ['kawah', 'putih', 'wisata', 'alam', 'bandung', 'sangat', 'indah', 'populer']



In [8]:
# Apply preprocessing ke semua deskripsi
print('Memproses 331 deskripsi destinasi (mungkin butuh 1-2 menit)...')
df['processed'] = df['Description'].apply(preprocess_text)
print('Preprocessing selesai')

# Tampilkan contoh hasil preprocessing
print('\n=== CONTOH HASIL PREPROCESSING ===')
for i in range(3):
    print(f'\n{df.iloc[i]["Place_Name"]}:')
    print(f'  Asli      : {df.iloc[i]["Description"]}')
    print(f'  Processed : {df.iloc[i]["processed"]}')

Memproses 331 deskripsi destinasi (mungkin butuh 1-2 menit)...
Preprocessing selesai

=== CONTOH HASIL PREPROCESSING ===

Alun-Alun Bandung:
  Asli      : Alun-alun pusat kota Bandung dengan taman rumput sintetis luas, area bersantai keluarga, dan ikon ruang publik perkotaan.
  Processed : alunalun pusat kota bandung taman rumput sintetis luas area santa keluarga ikon ruang publik kota

Gedung Sate:
  Asli      : Bangunan bersejarah peninggalan kolonial Belanda dengan arsitektur khas Indo-Eropa, kini menjadi kantor gubernur dan situs sejarah Kota Bandung.
  Processed : bangun sejarah tinggal kolonial belanda arsitektur khas indoeropa kini jadi kantor gubernur situs sejarah kota bandung

Jalan Braga:
  Asli      : Jalan bersejarah dengan deretan bangunan kolonial klasik, kafe heritage, dan galeri seni di pusat sejarah Kota Bandung.
  Processed : jalan sejarah deret bangun kolonial klasik kafe heritage galeri seni pusat sejarah kota bandung


## 4. Pembobotan TF-IDF

**TF (Term Frequency)**: frekuensi kemunculan kata dalam dokumen  
**IDF (Inverse Document Frequency)**: log(N / df) - mengukur kelangkaan kata  
**TF-IDF = TF x IDF**

Output: matrix berukuran (331, N) di mana N = jumlah unique terms

In [9]:
# Inisialisasi 
tfidf_vectorizer = TfidfVectorizer()

# Fit & transform teks yang sudah preprocessed
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed'])

print(f'TF-IDF Matrix shape: {tfidf_matrix.shape}')
print(f'  -> Jumlah dokumen     : {tfidf_matrix.shape[0]}')
print(f'  -> Jumlah unique term : {tfidf_matrix.shape[1]}')
print(f'\nContoh vocabulary (20 pertama):')
print(tfidf_vectorizer.get_feature_names_out()[:20])

TF-IDF Matrix shape: (331, 584)
  -> Jumlah dokumen     : 331
  -> Jumlah unique term : 584

Contoh vocabulary (20 pertama):
['acara' 'ada' 'adat' 'afrika' 'agam' 'agama' 'air' 'ajak' 'ajar'
 'aksesoris' 'aktif' 'aktivitas' 'ala' 'alam' 'alami' 'alir' 'alunalun'
 'amerika' 'anak' 'anakanak']


In [10]:
# Tampilkan contoh bobot TF-IDF untuk satu destinasi
sample_idx = 0
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix[sample_idx].toarray().flatten()

# Ambil kata dengan bobot tertinggi
top_indices = tfidf_scores.argsort()[::-1][:10]

print(f'=== BOBOT TF-IDF: {df.iloc[sample_idx]["Place_Name"]} ===')
print(f'Deskripsi: {df.iloc[sample_idx]["Description"]}\n')
print('Kata dengan bobot tertinggi:')
for idx in top_indices:
    if tfidf_scores[idx] > 0:
        print(f'  {feature_names[idx]:25s} -> {tfidf_scores[idx]:.4f}')

=== BOBOT TF-IDF: Alun-Alun Bandung ===
Deskripsi: Alun-alun pusat kota Bandung dengan taman rumput sintetis luas, area bersantai keluarga, dan ikon ruang publik perkotaan.

Kata dengan bobot tertinggi:
  sintetis                  -> 0.3670
  rumput                    -> 0.3670
  kota                      -> 0.3427
  ikon                      -> 0.3253
  santa                     -> 0.3119
  alunalun                  -> 0.3119
  luas                      -> 0.2703
  publik                    -> 0.2421
  ruang                     -> 0.2318
  pusat                     -> 0.1969


In [11]:
# ============================================================
# VERIFIKASI ANGKA UNTUK LAPORAN BAB IV (Tabel 4.3 & 4.4)
# ============================================================

# 1. Info Matriks TF-IDF (untuk Tabel 4.3)
print("=" * 50)
print("INFO MATRIKS TF-IDF (untuk Tabel 4.3)")
print("=" * 50)
print(f"Jumlah Dokumen (Destinasi) : {tfidf_matrix.shape[0]}")
print(f"Jumlah Unique Terms        : {tfidf_matrix.shape[1]}")
print(f"Total Elemen Matriks       : {tfidf_matrix.shape[0] * tfidf_matrix.shape[1]:,}")

sparsity = (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100
print(f"Sparsity Matriks           : {sparsity:.2f}%")

# 2. Bobot TF-IDF untuk Museum Geologi (untuk Tabel 4.4)
print("\n" + "=" * 50)
print("BOBOT TF-IDF: MUSEUM GEOLOGI (untuk Tabel 4.4)")
print("=" * 50)

mg_idx = df[df['Place_Name'] == 'Museum Geologi'].index[0]
print(f"Deskripsi: {df.iloc[mg_idx]['Description']}")
print(f"Processed: {df.iloc[mg_idx]['processed']}")
print()

scores = tfidf_matrix[mg_idx].toarray().flatten()
vocab = tfidf_vectorizer.get_feature_names_out()

print(f"{'No':3s} | {'Kata':20s} | {'Bobot TF-IDF'}")
print("-" * 45)
for no, idx in enumerate(scores.argsort()[::-1][:12], 1):
    if scores[idx] > 0:
        print(f"{no:3d} | {vocab[idx]:20s} | {scores[idx]:.4f}")

INFO MATRIKS TF-IDF (untuk Tabel 4.3)
Jumlah Dokumen (Destinasi) : 331
Jumlah Unique Terms        : 584
Total Elemen Matriks       : 193,304
Sparsity Matriks           : 98.32%

BOBOT TF-IDF: MUSEUM GEOLOGI (untuk Tabel 4.4)
Deskripsi: Museum edukasi yang menyimpan koleksi batuan, fosil purba, mineral, dan informasi sejarah geologi Indonesia.
Processed: museum edukasi simpan koleksi batu fosil purba mineral informasi sejarah geologi indonesia

No  | Kata                 | Bobot TF-IDF
---------------------------------------------
  1 | fosil                | 0.3602
  2 | mineral              | 0.3602
  3 | informasi            | 0.3363
  4 | geologi              | 0.3363
  5 | indonesia            | 0.2955
  6 | purba                | 0.2955
  7 | simpan               | 0.2955
  8 | museum               | 0.2455
  9 | batu                 | 0.2415
 10 | koleksi              | 0.2216
 11 | edukasi              | 0.2189
 12 | sejarah              | 0.1915


## 5. Perhitungan Cosine Similarity

Cosine Similarity mengukur kemiripan antara 2 vektor.  
Nilai antara **0 (tidak mirip)** sampai **1 (identik)**.

Output: matrix kemiripan berukuran (331, 331)

In [12]:
# Hitung Cosine Similarity untuk semua pasangan destinasi
cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f'Cosine Similarity Matrix shape: {cosine_sim_matrix.shape}')
print(f'\nContoh nilai similarity (5x5 pertama):')
print(pd.DataFrame(cosine_sim_matrix[:5, :5], 
                    index=df['Place_Name'][:5], 
                    columns=df['Place_Name'][:5]).round(3))

Cosine Similarity Matrix shape: (331, 331)

Contoh nilai similarity (5x5 pertama):
Place_Name            Alun-Alun Bandung  Gedung Sate  Jalan Braga  \
Place_Name                                                          
Alun-Alun Bandung                 1.000        0.065        0.110   
Gedung Sate                       0.065        1.000        0.302   
Jalan Braga                       0.110        0.302        1.000   
Trans Studio Bandung              0.075        0.015        0.017   
Saung Angklung Udjo               0.036        0.042        0.093   

Place_Name            Trans Studio Bandung  Saung Angklung Udjo  
Place_Name                                                       
Alun-Alun Bandung                    0.075                0.036  
Gedung Sate                          0.015                0.042  
Jalan Braga                          0.017                0.093  
Trans Studio Bandung                 1.000                0.000  
Saung Angklung Udjo                  

## 6. Function Rekomendasi

In [13]:
def get_recommendations_by_place(place_name, top_n=5):
    """
    Cari destinasi mirip berdasarkan nama destinasi (untuk halaman detail).
    """
    # Cari index destinasi
    matches = df[df['Place_Name'].str.lower() == place_name.lower()]
    if len(matches) == 0:
        return None
    
    idx = matches.index[0]
    
    # Ambil score similarity
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip yang pertama (diri sendiri), ambil top N
    sim_scores = sim_scores[1:top_n+1]
    
    # Buat dataframe hasil
    results = []
    for i, score in sim_scores:
        results.append({
            'Place_Name': df.iloc[i]['Place_Name'],
            'Category': df.iloc[i]['Category'],
            'City': df.iloc[i]['City'],
            'Similarity': round(score, 4)
        })
    return pd.DataFrame(results)


def get_recommendations_by_text(query_text, top_n=6):
    """
    Cari destinasi berdasarkan input teks user (untuk fitur pencarian).
    """
    # Preprocess query
    query_processed = preprocess_text(query_text)
    
    # Transform query ke vektor TF-IDF (pakai vectorizer yang sama)
    query_vector = tfidf_vectorizer.transform([query_processed])
    
    # Hitung similarity query vs semua dokumen
    sim_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Sort & ambil top N
    top_indices = sim_scores.argsort()[::-1][:top_n]
    
    results = []
    for i in top_indices:
        if sim_scores[i] > 0:
            results.append({
                'Place_Name': df.iloc[i]['Place_Name'],
                'Category': df.iloc[i]['Category'],
                'City': df.iloc[i]['City'],
                'Similarity': round(sim_scores[i], 4)
            })
    return pd.DataFrame(results)

print('Function rekomendasi siap')

Function rekomendasi siap


## 7. Test Rekomendasi

In [14]:
# TEST 1: Rekomendasi berdasarkan destinasi (Kawah Putih)
print('=== TEST 1: Destinasi serupa dengan KAWAH PUTIH ===')
result = get_recommendations_by_place('Kawah Putih', top_n=6)
print(result)

=== TEST 1: Destinasi serupa dengan KAWAH PUTIH ===
                Place_Name     Category               City  Similarity
0             Kawah Cibuni  Wisata Alam  Kabupaten Bandung      0.8867
1   Kawah Cibuni Rengganis  Wisata Alam  Kabupaten Bandung      0.8710
2             Kawah Wayang  Wisata Alam  Kabupaten Bandung      0.8465
3            Gunung Patuha  Wisata Alam  Kabupaten Bandung      0.1464
4        Sungai Palayangan  Wisata Alam  Kabupaten Bandung      0.0767
5  Gunung Tangkuban Perahu  Wisata Alam      Bandung Barat      0.0712


In [15]:
# TEST 2: Rekomendasi berdasarkan destinasi (Gunung Tangkuban Perahu)
print('=== TEST 2: Destinasi serupa dengan GUNUNG TANGKUBAN PERAHU ===')
result = get_recommendations_by_place('Gunung Tangkuban Perahu', top_n=6)
print(result)

=== TEST 2: Destinasi serupa dengan GUNUNG TANGKUBAN PERAHU ===
             Place_Name     Category               City  Similarity
0          Gunung Windu  Wisata Alam  Kabupaten Bandung      0.5762
1         Gunung Wayang  Wisata Alam  Kabupaten Bandung      0.5385
2       Gunung Artapela  Wisata Alam  Kabupaten Bandung      0.4992
3         Gunung Patuha  Wisata Alam  Kabupaten Bandung      0.4860
4  Gunung Putri Lembang  Wisata Alam      Bandung Barat      0.4640
5     Gunung Burangrang  Wisata Alam      Bandung Barat      0.4365


In [16]:
# TEST 3: Pencarian dengan keyword 'kawah'
print('=== TEST 3: Pencarian dengan keyword "kawah" ===')
result = get_recommendations_by_text('kawah', top_n=6)
print(result)

=== TEST 3: Pencarian dengan keyword "kawah" ===
               Place_Name     Category               City  Similarity
0            Kawah Cibuni  Wisata Alam  Kabupaten Bandung      0.8492
1            Kawah Wayang  Wisata Alam  Kabupaten Bandung      0.8378
2  Kawah Cibuni Rengganis  Wisata Alam  Kabupaten Bandung      0.8341
3             Kawah Putih  Wisata Alam  Kabupaten Bandung      0.8134


In [17]:
# TEST 4: Pencarian dengan keyword 'air terjun'
print('=== TEST 4: Pencarian dengan keyword "air terjun" ===')
result = get_recommendations_by_text('air terjun', top_n=6)
print(result)

=== TEST 4: Pencarian dengan keyword "air terjun" ===
                  Place_Name     Category               City  Similarity
0              Curug Cipanas  Wisata Alam      Bandung Barat      0.5481
1        Air Terjun Marbella  Wisata Alam      Bandung Barat      0.5041
2  Curug Bentang Padjadjaran  Wisata Alam  Kabupaten Bandung      0.4733
3             Curug Cinulang  Wisata Alam  Kabupaten Bandung      0.4668
4            Curug Panganten  Wisata Alam  Kabupaten Bandung      0.4554
5        Curug Sawer Cililin  Wisata Alam      Bandung Barat      0.4530


In [18]:
# TEST 5: Pencarian dengan keyword 'museum'
print('=== TEST 5: Pencarian dengan keyword "museum" ===')
result = get_recommendations_by_text('museum', top_n=6)
print(result)

=== TEST 5: Pencarian dengan keyword "museum" ===
                      Place_Name Category          City  Similarity
0     Museum Pendidikan Nasional   Museum  Kota Bandung      0.3364
1             Museum Asia Afrika   Museum  Kota Bandung      0.2818
2           Museum Pos Indonesia   Museum  Kota Bandung      0.2786
3            Museum Kota Bandung   Museum  Kota Bandung      0.2782
4  Museum Konferensi Asia Afrika   Museum  Kota Bandung      0.2749
5              Museum Sri Baduga   Museum  Kota Bandung      0.2664


In [19]:
# TEST 6: Pencarian dengan keyword 'pemandian air panas'
print('=== TEST 6: Pencarian dengan keyword "pemandian air panas" ===')
result = get_recommendations_by_text('pemandian air panas', top_n=6)
print(result)

=== TEST 6: Pencarian dengan keyword "pemandian air panas" ===
                          Place_Name     Category               City  \
0         Pemandian Air Panas Walini  Wisata Alam  Kabupaten Bandung   
1       Pemandian Air Panas Cibolang  Wisata Alam  Kabupaten Bandung   
2  Pemandian Air Panas Nagrak Tengah  Wisata Alam      Bandung Barat   
3        Pemandian Air Panas Cipanas  Wisata Alam      Bandung Barat   
4                Maribaya Hot Spring  Wisata Alam  Kabupaten Bandung   
5                Cibolang Hot Spring  Wisata Alam       Kota Bandung   

   Similarity  
0      0.6468  
1      0.6436  
2      0.6421  
3      0.6383  
4      0.5331  
5      0.5085  


## 8. Save Model (.pkl)

Simpan model agar bisa di-load oleh backend Flask tanpa training ulang.  
File yang disimpan:
- `tfidf_vectorizer.pkl` - vectorizer untuk transform query baru
- `tfidf_matrix.pkl` - matrix TF-IDF dari 331 destinasi
- `cosine_sim_matrix.pkl` - matrix similarity 331x331
- `dataset_processed.pkl` - dataset dengan kolom processed

In [20]:
# Buat folder model kalau belum ada
os.makedirs('../model', exist_ok=True)

# Save semua file
with open('../model/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print('tfidf_vectorizer.pkl saved')

with open('../model/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)
print('tfidf_matrix.pkl saved')

with open('../model/cosine_sim_matrix.pkl', 'wb') as f:
    pickle.dump(cosine_sim_matrix, f)
print('cosine_sim_matrix.pkl saved')

with open('../model/dataset_processed.pkl', 'wb') as f:
    pickle.dump(df, f)
print('dataset_processed.pkl saved')

print('\nSemua model berhasil disimpan di folder ../model/')
print('Sekarang restart backend Flask: python app.py')

tfidf_vectorizer.pkl saved
tfidf_matrix.pkl saved
cosine_sim_matrix.pkl saved
dataset_processed.pkl saved

Semua model berhasil disimpan di folder ../model/
Sekarang restart backend Flask: python app.py


## 9. Verifikasi File Tersimpan

In [21]:
# Verifikasi semua file tersimpan dengan benar
print('=== FILE DI FOLDER MODEL ===')
for filename in os.listdir('../model'):
    filepath = os.path.join('../model', filename)
    size_kb = os.path.getsize(filepath) / 1024
    print(f'  {filename:30s} ({size_kb:.1f} KB)')

print('\nNOTEBOOK SELESAI!')
print('Selanjutnya: cd ../backend && python app.py')

=== FILE DI FOLDER MODEL ===
  cosine_sim_matrix.pkl          (856.1 KB)
  dataset_processed.pkl          (67.2 KB)
  README.txt                     (0.1 KB)
  tfidf_matrix.pkl               (39.7 KB)
  tfidf_vectorizer.pkl           (11.9 KB)

NOTEBOOK SELESAI!
Selanjutnya: cd ../backend && python app.py
